# Algorithm 5.4 — State vector from a radar observation

**Goal:** turn what a tracking station actually measures — **slant range** $\rho$, **azimuth** $A$, **elevation** $a$, and their rates $\dot\rho,\dot A,\dot a$, taken at a known site (local sidereal time $\theta$, latitude $\phi$, altitude $H$) — into the object's full geocentric state vector $(\mathbf{r},\mathbf{v})$.

**Code (answer key):** [`rv_from_observe`](../curtis/algorithms/alg_5_4_rv_from_observe.py) · **Book:** §5.8, Algorithm 5.4 · **Worked example:** 5.10

This is where the LST from 5.3 gets used. No iteration — just a long, careful chain of spherical trigonometry.

## Read first

| Symbol | Meaning |
|---|---|
| $\rho,\dot\rho$ | slant range (km) and its rate (km/s) |
| $A,\dot A$ | azimuth (deg, clockwise from north) and rate |
| $a,\dot a$ | elevation above the horizon (deg) and rate |
| $\theta$ | local sidereal time of the site (deg) — from Algorithm 5.3 |
| $\phi, H$ | geodetic latitude (deg) and altitude (km) of the site |
| $\mathbf{R},\dot{\mathbf{R}}$ | site position and (rotation) velocity |
| $\delta, h, \alpha$ | topocentric declination, hour angle, right ascension |
| $\hat{\boldsymbol{\rho}}$ | unit vector from site to object |

**Core relation (Eq 5.63):** $\;\mathbf{r}=\mathbf{R}+\rho\,\hat{\boldsymbol{\rho}}\;$ — geocentric position = site position + slant-range vector.

## The picture (site + slant range = orbit position)

A radar sees an object *relative to itself*: how far ($\rho$), and in which direction (azimuth $A$ around the horizon, elevation $a$ above it). To put that object on an orbit you must re-express it from **Earth's centre**:
$$\mathbf{r}=\underbrace{\mathbf{R}}_{\text{centre}\to\text{site}}+\underbrace{\rho\,\hat{\boldsymbol{\rho}}}_{\text{site}\to\text{object}}.$$

Three sub-problems:

1. **Where is the site?** $\mathbf{R}$ from latitude, altitude and LST — using the *oblate* Earth (it is not a sphere).
2. **Which way is the object?** Convert the local horizon angles $(A,a)$ into the inertial-frame direction $\hat{\boldsymbol{\rho}}$ (via declination $\delta$ and right ascension $\alpha$).
3. **Velocity** adds three pieces: the site's motion as Earth spins ($\dot{\mathbf{R}}=\boldsymbol{\omega}\times\mathbf{R}$), the range closing/opening ($\dot\rho\,\hat{\boldsymbol{\rho}}$), and the direction sweeping ($\rho\,\dot{\hat{\boldsymbol{\rho}}}$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc

In [ ]:
# Side-view schematic of r = R + rho * rho_hat  (illustrative geometry).
phi_s = np.radians(62)                     # where the site sits on the drawn Earth
S = np.array([np.cos(phi_s), np.sin(phi_s)])           # site on the surface
zenith = S / np.linalg.norm(S)                          # local "up"
horiz = np.array([-zenith[1], zenith[0]])               # local horizon direction
el = np.radians(45)                                     # elevation above horizon
look = np.cos(el)*horiz + np.sin(el)*zenith             # slant-range direction
P = S + 1.25*look                                       # the object

fig, ax = plt.subplots(figsize=(6.6, 6.6))
circ = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(circ), np.sin(circ), color='#c2c0b6', lw=1.2)        # Earth
ax.plot([S[0]-0.55*horiz[0], S[0]+0.85*horiz[0]],
        [S[1]-0.55*horiz[1], S[1]+0.85*horiz[1]], color='#888780', lw=1, ls='--')  # horizon
ax.annotate('', xy=S, xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='#1D9E75', lw=2))
ax.annotate('', xy=P, xytext=tuple(S), arrowprops=dict(arrowstyle='-|>', color='#D85A30', lw=2))
ax.annotate('', xy=P, xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='#D4537E', lw=2))
ax.add_patch(Arc(S, 0.55, 0.55, angle=np.degrees(np.arctan2(horiz[1], horiz[0])),
                 theta1=0, theta2=45, color='#D85A30', lw=1.8))
ax.plot(0, 0, 'o', color='#3B8BD4', ms=8)
ax.plot(*S, 'o', color='#1D9E75', ms=6)
ax.plot(*P, 'o', color='#D4537E', ms=7)

ax.annotate('Earth centre', (0, 0), textcoords='offset points', xytext=(6, -13), color='#3B8BD4')
ax.annotate('R (site)', 0.5*S + np.array([0.10, 0.0]), color='#1D9E75')
ax.annotate(r'$\rho\,\hat\rho$ (slant range)', S + 0.55*look + np.array([0.08, -0.02]), color='#D85A30')
ax.annotate('r (object)', 0.55*P + np.array([-0.62, 0.05]), color='#D4537E')
ax.annotate('object', P + np.array([0.06, 0.04]), color='#D4537E', fontsize=10)
ax.annotate('a', S + 0.34*(np.cos(el/2)*horiz + np.sin(el/2)*zenith), color='#D85A30', fontsize=11)
ax.annotate('horizon', S + 0.85*horiz + np.array([0, 0.05]), color='#888780', fontsize=9)

ax.set_aspect('equal'); ax.axis('off')
ax.set_xlim(-1.7, 1.7); ax.set_ylim(-1.35, 2.4)
ax.set_title('A radar measures (rho, A, a) from the site; r = R + rho * rho_hat')
plt.show()

## See it — re-base the measurement to Earth's centre

The station measures the orange slant-range vector $\rho\hat{\boldsymbol{\rho}}$ (its length is $\rho$, its tilt above the dashed horizon is the elevation $a$, and its compass bearing — into/out of the page here — is the azimuth $A$). Adding the green site vector $\mathbf{R}$ gives the pink geocentric position $\mathbf{r}$. The whole algorithm is making those two vectors precise in the inertial frame, then differentiating once for $\mathbf{v}$.

## Where these equations come from

### Site position on an oblate Earth (Eq 5.56)
The Earth is flattened, so the distance from the centre to the surface depends on latitude. With flattening $f$ and equatorial radius $R_e$,
$$\mathbf{R}=\Big[\big(\tfrac{R_e}{\sqrt{1-(2f-f^2)\sin^2\phi}}+H\big)\cos\phi\cos\theta,\ \big(\dots\big)\cos\phi\sin\theta,\ \big(\tfrac{R_e(1-f)^2}{\sqrt{1-(2f-f^2)\sin^2\phi}}+H\big)\sin\phi\Big].$$
The two different prefactors (note the $(1-f)^2$ on the $z$-component) are what make it an ellipsoid rather than a sphere. The LST $\theta$ supplies the longitude orientation in the inertial frame.

### Look angles → direction in the sky (Eqs 5.83)
Spherical trig on the observer's triangle converts horizon angles $(A,a)$ at latitude $\phi$ into equatorial coordinates:
$$\delta=\sin^{-1}(\cos\phi\cos A\cos a+\sin\phi\sin a)\ \ (\text{declination}),$$
$$h=\cos^{-1}\!\Big(\tfrac{\cos\phi\sin a-\sin\phi\cos A\cos a}{\cos\delta}\Big)\ \ (\text{hour angle}),\qquad \alpha=\theta-h\ \ (\text{right ascension}).$$
The quadrant of $h$ is fixed by whether the object is rising or setting ($0<A<\pi$). Then the unit direction is $\hat{\boldsymbol{\rho}}=[\cos\alpha\cos\delta,\ \sin\alpha\cos\delta,\ \sin\delta]$ (Eq 5.57).

### Position and velocity (Eqs 5.63, 5.64, 5.66, 5.84, 5.85)
Position is $\mathbf{r}=\mathbf{R}+\rho\hat{\boldsymbol{\rho}}$. Velocity differentiates each piece: the site swings with the Earth, $\dot{\mathbf{R}}=\boldsymbol{\omega}\times\mathbf{R}$ with $\boldsymbol{\omega}=[0,0,\omega_E]$; the range changes as $\dot\rho\hat{\boldsymbol{\rho}}$; and the line of sight rotates as $\rho\dot{\hat{\boldsymbol{\rho}}}$, where $\dot{\hat{\boldsymbol{\rho}}}$ needs $\dot\delta$ (Eq 5.84) and $\dot\alpha$ (Eq 5.85), built from the measured angle rates $\dot A,\dot a$. Summed: $\mathbf{v}=\dot{\mathbf{R}}+\dot\rho\hat{\boldsymbol{\rho}}+\rho\dot{\hat{\boldsymbol{\rho}}}$.

## Step by step (in code order)

It is a long transcription — work top to bottom; nothing loops.

**Setup:** `deg = pi/180`; `omega = [0,0,wE]`; convert $A,\dot A,a,\dot a,\theta,\phi$ to radians.

**Site (Eq 5.56, 5.66):** build `fac1`, `fac2` (the two oblate prefactors), then $\mathbf{R}$, then $\dot{\mathbf{R}}=\boldsymbol\omega\times\mathbf{R}$.

**Direction (Eqs 5.83, 5.57):** `dec`, then `h` (flip to $2\pi-h$ if $0<A<\pi$), then `RA = theta - h`, then $\hat{\boldsymbol\rho}$.

**Position (Eq 5.63):** $\mathbf{r}=\mathbf{R}+\rho\,\hat{\boldsymbol\rho}$.

**Rates (Eqs 5.84, 5.85, 5.69/5.72):** `decdot`, `RAdot`, then $\dot{\hat{\boldsymbol\rho}}$.

**Velocity (Eq 5.64):** $\mathbf{v}=\dot{\mathbf{R}}+\dot\rho\,\hat{\boldsymbol\rho}+\rho\,\dot{\hat{\boldsymbol\rho}}$.

**↓ Now type the implementation below.** It is the longest one yet, but it is pure formula transcription (`np.sin/cos/tan`, `np.cross`). Take the blocks one at a time and check against the comments. Answer key linked above; peek only after you try.

In [ ]:
def rv_from_observe(rho, rhodot, A, Adot, a, adot, theta, phi, H,
                    f=1/298.256421867, Re=6378.13655, wE=7.292115e-5):
    """Geocentric state vector (r, v) from a radar observation. Angles in degrees."""
    deg = np.pi/180
    omega = np.array([0.0, 0.0, wE])

    # Convert angular quantities to radians:
    #   A, Adot, a, adot, theta, phi  *= deg

    # --- Site position & velocity ---
    # Eq 5.56:
    #   fac1 = Re / sqrt(1 - (2*f - f*f)*sin(phi)**2)
    #   fac2 = Re*(1-f)**2 / sqrt(1 - (2*f - f*f)*sin(phi)**2)
    #   R = [ (fac1+H)*cos(phi)*cos(theta),
    #         (fac1+H)*cos(phi)*sin(theta),
    #         (fac2+H)*sin(phi) ]
    # Eq 5.66:
    #   Rdot = cross(omega, R)

    # --- Direction to the object ---
    # Eq 5.83a:  dec = asin(cos(phi)*cos(A)*cos(a) + sin(phi)*sin(a))
    # Eq 5.83b:  h = acos((cos(phi)*sin(a) - sin(phi)*cos(A)*cos(a)) / cos(dec))
    #            if 0 < A < pi:  h = 2*pi - h
    # Eq 5.83c:  RA = theta - h
    # Eq 5.57:   Rho = [cos(RA)*cos(dec), sin(RA)*cos(dec), sin(dec)]

    # --- Position (Eq 5.63) ---
    #   r = R + rho*Rho

    # --- Rates ---
    # Eq 5.84:  decdot = (-Adot*cos(phi)*sin(A)*cos(a)
    #                     + adot*(sin(phi)*cos(a) - cos(phi)*cos(A)*sin(a))) / cos(dec)
    # Eq 5.85:  RAdot = wE + (Adot*cos(A)*cos(a) - adot*sin(A)*sin(a)
    #                         + decdot*sin(A)*cos(a)*tan(dec)) \
    #                        / (cos(phi)*sin(a) - sin(phi)*cos(A)*cos(a))
    # Eqs 5.69, 5.72:
    #   Rhodot = [-RAdot*sin(RA)*cos(dec) - decdot*cos(RA)*sin(dec),
    #              RAdot*cos(RA)*cos(dec) - decdot*sin(RA)*sin(dec),
    #              decdot*cos(dec)]

    # --- Velocity (Eq 5.64) ---
    #   v = Rdot + rhodot*Rho + rho*Rhodot

    # return r, v
    raise NotImplementedError("transcribe the blocks above, then delete this line")

## Verify — Example 5.10

**Input:** $\rho=2551$ km, $\dot\rho=0$, $A=90°$, $\dot A=0.1130°/s$, $a=30°$, $\dot a=0.05651°/s$, LST $\theta=300°$, latitude $\phi=60°$, $H=0$. Earth: $f=1/298.256421867$, $R_e=6378.13655$ km, $\omega_E=7.292115\times10^{-5}$ rad/s.

**Expected:** $\mathbf{r}=[3830.68,\,-2216.47,\,6605.09]$ km, $\mathbf{v}=[1.50357,\,-4.56099,\,-0.291536]$ km/s.

Run once your function is typed.

In [ ]:
r, v = rv_from_observe(rho=2551.0, rhodot=0.0, A=90.0, Adot=0.1130,
                       a=30.0, adot=0.05651, theta=300.0, phi=60.0, H=0.0)

print(f"r (km)   = [{r[0]:.6g} {r[1]:.6g} {r[2]:.6g}]   (expect [3830.68  -2216.47  6605.09])")
print(f"v (km/s) = [{v[0]:.6g} {v[1]:.6g} {v[2]:.6g}]   (expect [1.50357  -4.56099  -0.291536])")

assert np.allclose(r, [3830.68, -2216.47, 6605.09], atol=1e-2)
assert np.allclose(v, [1.50357, -4.56099, -0.291536], atol=1e-4)
print("\nAll checks passed ✔")

## What this confirms

- A raw radar measurement becomes a usable orbit state by **re-basing it to Earth's centre**: $\mathbf{r}=\mathbf{R}+\rho\hat{\boldsymbol\rho}$, then differentiating for $\mathbf{v}$.
- The **oblate** Earth matters: the site vector uses the ellipsoid, not a sphere.
- **LST is the glue** — it orients the site (and hence the right ascension) in the inertial frame, which is exactly what 5.3 produced.

**Next:** Algorithms 5.5/5.6 — **Gauss's method** of angles-only orbit determination with iterative improvement, the last and heaviest of Chapter 5. It chains three of these line-of-sight directions with `kepler_U` and the Lagrange coefficients to recover an orbit from angle measurements alone.